In [1]:
set.seed(405)
options(stringsAsFactors = FALSE)

In [2]:
required_packages <- c("dplyr", "readr", "tibble", "tidyr", "stringr")
missing_packages <- required_packages[!vapply(required_packages, requireNamespace, logical(1), quietly = TRUE)]
if (length(missing_packages) > 0) install.packages(missing_packages, repos = "https://cloud.r-project.org")
invisible(lapply(required_packages, library, character.only = TRUE))

Warning message:
“package ‘dplyr’ was built under R version 4.4.3”

Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Warning message:
“package ‘readr’ was built under R version 4.4.3”
Warning message:
“package ‘tibble’ was built under R version 4.4.3”


In [3]:
diag_m1 <- readr::read_csv("../output/model1_nuts/diagnostics_summary.csv", show_col_types = FALSE) %>%
  tidyr::pivot_wider(names_from = metric, values_from = value) %>%
  dplyr::mutate(model = "Model1")

diag_m2 <- readr::read_csv("../output/model2_nuts/diagnostics_summary.csv", show_col_types = FALSE) %>%
  tidyr::pivot_wider(names_from = metric, values_from = value) %>%
  dplyr::mutate(model = "Model2")

nuts_diag_summary <- dplyr::bind_rows(diag_m1, diag_m2) %>%
  dplyr::select(model, dplyr::everything())
nuts_diag_summary

model,max_rhat,min_ess,n_params_rhat_gt_1.01,n_divergences,max_treedepth
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
Model1,1.002933,1394.4297,0,NA,NA
Model2,1.003375,996.4545,0,0,8


In [4]:
vi_compare <- readr::read_csv("../output/vi/compare_metrics.csv", show_col_types = FALSE)
loo_summary <- readr::read_csv("../output/model_compare/loo_summary.csv", show_col_types = FALSE)
loo_diff <- readr::read_csv("../output/model_compare/loo_diff.csv", show_col_types = FALSE)
sens_metrics <- readr::read_csv("../output/sensitivity/sensitivity_metrics.csv", show_col_types = FALSE)
recovery_metrics <- readr::read_csv("../output/validation/recovery_metrics.csv", show_col_types = FALSE)

vi_compare

model,n_parameters,median_abs_mean_diff,median_sd_ratio_vi_to_nuts,p10_sd_ratio_vi_to_nuts,p90_sd_ratio_vi_to_nuts
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
Model1,13,0.008539536,0.6648920,0.31350795,0.9379152
Model2,17,0.036147343,0.5723045,0.07070987,0.8530557


In [5]:
final_summary <- dplyr::bind_rows(
  nuts_diag_summary %>%
    dplyr::transmute(section = "NUTS diagnostics", key = model, value = paste0("max_rhat=", round(max_rhat, 4), ", min_ess=", round(min_ess, 1), ", divergences=", ifelse(is.na(n_divergences), 0, n_divergences))),
  vi_compare %>%
    dplyr::transmute(section = "VI vs NUTS", key = model, value = paste0("median_sd_ratio=", round(median_sd_ratio_vi_to_nuts, 3), ", median_abs_mean_diff=", round(median_abs_mean_diff, 3))),
  loo_summary %>%
    dplyr::transmute(section = "Model comparison", key = model, value = paste0("elpd_loo=", round(elpd_loo, 2), ", se=", round(se_elpd_loo, 2))),
  loo_diff %>%
    dplyr::transmute(section = "Model comparison", key = metric, value = round(value, 3) %>% as.character()),
  sens_metrics %>%
    dplyr::transmute(section = "Prior sensitivity", key = paste(model, prior, sep = "_"), value = paste0("median_abs_mean_shift=", round(median_abs_mean_shift, 3), ", median_sd_ratio=", round(median_sd_ratio_vs_base, 3))),
  recovery_metrics %>%
    dplyr::transmute(section = "Synthetic recovery", key = model, value = paste0("coverage_rate=", round(coverage_rate, 3), ", median_abs_error=", round(median_abs_error, 3), ", max_rhat=", round(max_rhat, 3)))
)
final_summary

section,key,value
<chr>,<chr>,<chr>
NUTS diagnostics,Model1,"max_rhat=1.0029, min_ess=1394.4, divergences=0"
NUTS diagnostics,Model2,"max_rhat=1.0034, min_ess=996.5, divergences=0"
VI vs NUTS,Model1,"median_sd_ratio=0.665, median_abs_mean_diff=0.009"
VI vs NUTS,Model2,"median_sd_ratio=0.572, median_abs_mean_diff=0.036"
Model comparison,Model2,"elpd_loo=-2136.73, se=24.11"
Model comparison,Model1,"elpd_loo=-2136.88, se=24.13"
Model comparison,elpd_diff_m2_minus_m1,0.153
Model comparison,se_diff_approx,34.111
Prior sensitivity,Model1_tight,"median_abs_mean_shift=0.005, median_sd_ratio=0.95"


In [6]:
artifact_manifest <- tibble::tribble(
  ~path, ~stage, ~description,
  "output/model1_nuts/diagnostics_summary.csv", "Model 1 NUTS", "Convergence and ESS diagnostics",
  "output/model2_nuts/diagnostics_summary.csv", "Model 2 NUTS", "Convergence, ESS, divergences, treedepth",
  "output/vi/compare_metrics.csv", "VI comparison", "Aggregate VI vs NUTS discrepancy metrics",
  "output/model_compare/loo_summary.csv", "Model comparison", "PSIS-LOO estimates for both models",
  "output/model_compare/loo_diff.csv", "Model comparison", "elpd difference and uncertainty",
  "output/model_compare/ppc_model1_bars.png", "Model comparison", "PPC bars for Model 1",
  "output/model_compare/ppc_model2_bars.png", "Model comparison", "PPC bars for Model 2",
  "output/sensitivity/sensitivity_metrics.csv", "Prior sensitivity", "Sensitivity metrics by model and prior",
  "output/validation/recovery_metrics.csv", "Validation", "Synthetic recovery metrics",
  "output/validation/recovery_scatter.png", "Validation", "True vs posterior mean recovery plot"
) %>%
  dplyr::mutate(exists = file.exists(file.path("..", path)))
artifact_manifest

path,stage,description,exists
<chr>,<chr>,<chr>,<lgl>
output/model1_nuts/diagnostics_summary.csv,Model 1 NUTS,Convergence and ESS diagnostics,TRUE
output/model2_nuts/diagnostics_summary.csv,Model 2 NUTS,"Convergence, ESS, divergences, treedepth",TRUE
output/vi/compare_metrics.csv,VI comparison,Aggregate VI vs NUTS discrepancy metrics,TRUE
output/model_compare/loo_summary.csv,Model comparison,PSIS-LOO estimates for both models,TRUE
output/model_compare/loo_diff.csv,Model comparison,elpd difference and uncertainty,TRUE
output/model_compare/ppc_model1_bars.png,Model comparison,PPC bars for Model 1,TRUE
output/model_compare/ppc_model2_bars.png,Model comparison,PPC bars for Model 2,TRUE
output/sensitivity/sensitivity_metrics.csv,Prior sensitivity,Sensitivity metrics by model and prior,TRUE
output/validation/recovery_metrics.csv,Validation,Synthetic recovery metrics,TRUE


In [7]:
dir.create("../output/final", recursive = TRUE, showWarnings = FALSE)
readr::write_csv(final_summary, "../output/final/final_summary_table.csv")
readr::write_csv(artifact_manifest, "../output/final/artifact_manifest.csv")